In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")


In [2]:
from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
data=loader.load()

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
docs=text_splitter.split_documents(data)




In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [5]:
from langchain_community.vectorstores import FAISS
db=FAISS.from_documents(docs,embeddings)

In [6]:
query="Many of these algorithms are insufficient for solving large reasoning problems "
retriever=db.as_retriever()
response=retriever.invoke(query)
response

[Document(id='cdd326ff-7f2b-4387-b0c0-977309ba18c0', metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='Reasoning and problem-solving\nEarly researchers developed algorithms that imitated step-by-step reasoning that humans use when they solve puzzles or make logical deductions.[11] By the late 1980s and 1990s, methods were developed for dealing with uncertain or incomplete information, employing concepts from probability and economics.[12]\nMany of these algorithms are insufficient for solving large reasoning problems because they experience a "combinatorial explosion": They become exponentially slower as the problems grow.[13] Even humans rarely use the step-by-step deduction that early AI research could model. They solve most of their problems using fast, intuitive judgments.[14] Accurate and efficient reasoning is an unsolved problem.'),
 Document(id='b0bab3fe-b5b5-4b37-8946-bf

In [7]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm_endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="conversational",
    temperature=0.7,
    max_new_tokens=512,
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [14]:
prompt=ChatPromptTemplate([
    ("system","You are a helpful assistant. Answer the question based ONLY on the given context. Keep the answer concise and to the point. Do not ask follow-up questions."),
    ("human","""
       context:
     {context}
     question:
     {input}
""")]
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain=({"context":retriever | format_docs ,"input": RunnablePassthrough()}
        | prompt 
        | llm
        | StrOutputParser()
        
        )

response=rag_chain.invoke('what is artifical intelligence')
response


'Define artificial intelligence in terms of external behavior, rather than internal structure or human-likeness, according to Russell and Norvig.'